In [1]:
from transcript_configurator import TranscriptConfigurator, ScoreboardCalculator, GPTConcurrentRequester
import os
import pandas as pd
import dtale
import spacy
import nltk

2023-06-22 00:34:41.825602: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
open_ai_keys = [os.getenv(f"OPENAI_KEY_{i}") for i in range(1, 26)]
pd.set_option('display.max_columns', 500)
nlp = spacy.load("en_core_web_lg")
english_vocab = set(w.lower() for w in nltk.corpus.words.words())

In [3]:
original_transcript_path = "../runs/2008-09-26-debate-preclean.txt"
known_speakers = {'LEHRER', 'OBAMA', 'MCCAIN'}

In [4]:
configurator = TranscriptConfigurator()
empty_scoreboard = configurator.get_updated_scoreboard(original_transcript_path, known_speakers)

In [5]:
scoreboard_calculator = ScoreboardCalculator(empty_scoreboard, nlp, english_vocab)

In [6]:
gpt_requester = GPTConcurrentRequester(empty_scoreboard, open_ai_keys, original_transcript_path, scoreboard_calculator)
final_scoreboard = gpt_requester.get_epl(startturn=0, endturn=5)

In [ ]:
x = dtale.show(final_scoreboard)
x.open_browser()

In [ ]:
import ast
from collections import OrderedDict

def parse_file(input_file):
    with open(input_file, 'r') as file:
        lines = file.readlines()
    
    data_dict = OrderedDict()
    for line in lines:
        # Split the line at the colon
        index, json_dict = line.split(":", 1)

        # Parse the dictionary using ast.literal_eval()
        json_dict = ast.literal_eval(json_dict.strip())

        # Add the index and dictionary to our OrderedDict
        data_dict[int(index)] = json_dict

    return data_dict

input_file = "../runs/epl-scores/2008-09-26-debate-preclean_withEPL_content.json"
parsed_data = parse_file(input_file)

len(parsed_data)


In [ ]:
import pandas as pd

# create an empty list to hold your dataframes
df_list = []

# iterate over ordered dictionary
for index, value in parsed_data.items():
    # create a dataframe from each dictionary and append it to the list
    df_list.append(pd.DataFrame(value, index=[index]))

# concatenate all the dataframes in the list
df = pd.concat(df_list)

df.sort_index()


df.columns

In [ ]:
# rename the columns in df
df.rename(columns={'E': 'ethos_score', 'P': 'pathos_score', 'L': 'logos_score'}, inplace=True)

# update the columns in empty_scoreboard with the columns in df
final_scoreboard = empty_scoreboard.copy()
final_scoreboard.update(df)


final_scoreboard

In [ ]:
# create an instance of ScoreboardCalculator
scoreboard_calculator = ScoreboardCalculator(final_scoreboard, nlp, english_vocab)

# call the update_scoreboard method
updated_final_scoreboard = scoreboard_calculator.update_scoreboard()

x = dtale.show(updated_final_scoreboard)




In [ ]:
x.open_browser()

In [ ]:
updated_final_scoreboard.columns

In [ ]:
updated_final_scoreboard.loc[321]